# 10a Prepare the SI Concept Corpus

Join screened pair-year PMID membership to the existing abstract cache and SI output. This notebook creates the fixed development/confirmation pair split and estimates labeling volume before any model calls.

The split is deterministic (20% confirmation, seed 42). Notebook 10b identifies articles shared by development and confirmation pairs and excludes them from concept-generation examples.

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "amr").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "amr").exists():
    raise RuntimeError(f"Could not find repository root from {Path.cwd()}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from amr.pipeline_config import cfg, cfg_path, load_pipeline_config
from amr.pipelines.concepts import build_si_concept_paths, load_concept_corpus, corpus_preflight
from amr.si.concepts import (
    SEED_CONCEPTS, concept_catalog_hash, estimate_classification_cost,
    classify_concept_abstracts, build_yearly_concept_signals,
    rolling_origin_concept_forecasts, summarise_incremental_forecast_gains,
    propose_concept_children,
)

PIPELINE_CONFIG = load_pipeline_config(project_root=PROJECT_ROOT)
CONCEPT_PATHS = build_si_concept_paths(PROJECT_ROOT)
MODEL_NAME = str(cfg(PIPELINE_CONFIG, "llm.model", "gpt-5-nano"))

YEARLY_SCREENED_PATH = cfg_path(PIPELINE_CONFIG, "paths.yearly_screened", PROJECT_ROOT)
ABSTRACT_CACHE_PATH = cfg_path(PIPELINE_CONFIG, "paths.abstract_cache", PROJECT_ROOT)
SI_PATH = PROJECT_ROOT / "outputs" / "parquets" / "yearly_si.parquet"
PATHOGEN_PRIORITY_PATH = cfg_path(PIPELINE_CONFIG, "paths.pathogen_priority", PROJECT_ROOT)
MIN_PATHOGENIC_INDEX = float(cfg(PIPELINE_CONFIG, "collection.min_pathogenic_index", 0))
NUM_PATHOGENS = 10  # Set an integer (e.g. 10) for a restricted run; all antibiotics are retained.
HELDOUT_FRACTION = 0.20
SPLIT_SEED = 42


## Load and save the corpus

### What this stage builds

The screened table defines article membership for each pathogen–antibiotic pair and year. The SI table restricts analysis to pair-years with an observed outcome; the abstract cache supplies text for each PMID. A paper can belong to several pairs, so the corpus retains one row per pair-year–PMID association, while the separate abstract table keeps one text record per PMID.

Set `NUM_PATHOGENS` before running this section. An integer takes the highest-ranked eligible pathogens available in both the SI and screened data; `None` keeps the full corpus. The limit is applied before splitting pairs, and all antibiotics for selected pathogens remain included. Rebuild the corpus and split whenever this setting changes.

In [2]:
corpus, pair_split = load_concept_corpus(
    screened_path=YEARLY_SCREENED_PATH,
    abstracts_path=ABSTRACT_CACHE_PATH,
    si_path=SI_PATH,
    heldout_fraction=HELDOUT_FRACTION,
    split_seed=SPLIT_SEED,
    pathogen_priority_path=PATHOGEN_PRIORITY_PATH,
    num_pathogens=NUM_PATHOGENS,
    min_pathogenic_index=MIN_PATHOGENIC_INDEX,
)
CONCEPT_PATHS["directory"].mkdir(parents=True, exist_ok=True)
unique_abstracts = corpus[["pmid", "title", "abstract", "abstract_available"]].drop_duplicates("pmid")
corpus.drop(columns=["title", "abstract"]).to_parquet(CONCEPT_PATHS["corpus"], index=False)
unique_abstracts.to_parquet(CONCEPT_PATHS["abstracts"], index=False)
pair_split.to_csv(CONCEPT_PATHS["pair_split"], index=False)
preflight = corpus_preflight(corpus)
print(f"Pathogen limit: {NUM_PATHOGENS}; selected pathogens: {corpus['pathogen'].nunique():,}; all antibiotics retained")
preflight

Pathogen limit: 10; selected pathogens: 10; all antibiotics retained


{'pair_year_pmid_associations': 137695,
 'unique_pmids': 55101,
 'unique_abstracts_to_classify': 55101,
 'abstract_available_associations': 137695,
 'abstract_coverage': 1.0,
 'development_pairs': 229,
 'confirmation_pairs': 57}

## Jev request-volume preflight

Jev sends one typed concept-presence question per catalog entry for each unique abstract. Review the selected corpus size before enabling classification.

The request count is the number of unique nonblank abstracts after any pathogen restriction. Jev receives one request per abstract, no matter how many pairs cite it, and each request carries a question for each catalog concept. This is a request-count estimate, not a monetary cost estimate. Check the count and abstract coverage before enabling classification in notebook 10c.

In [3]:
MODEL_NAME = str(cfg(PIPELINE_CONFIG, "system1.model", "typesafe/jev-1.13"))
print({"model": MODEL_NAME, "Jev requests for full run": preflight["unique_abstracts_to_classify"]})


{'model': 'typesafe/jev-1.13', 'Jev requests for full run': 55101}


## Data checks

These checks verify that exploding PMID lists did not create duplicate pair-year–article links and that both split partitions contain pairs. Review abstract coverage too: missing or blank text cannot contribute a classified score. Shared PMIDs may occur across the two pair partitions, but notebook 10b removes those records from concept-generation examples.

In [4]:
assert corpus[["pathogen", "antibiotic", "year", "pmid"]].duplicated().sum() == 0
assert set(pair_split["split"]) == {"development", "confirmation"}
assert pair_split.groupby("split")[["pathogen", "antibiotic"]].size().min() > 0
corpus.groupby("split").agg(
    pairs=("pathogen", "nunique"),
    pair_year_article_links=("pmid", "size"),
    unique_pmids=("pmid", "nunique"),
    abstract_coverage=("abstract_available", "mean"),
)

,pairs,pair_year_article_links,unique_pmids,abstract_coverage
split,,,,
confirmation,10,25846,20684,1.0
development,9,111849,50245,1.0
